# CS604 Colab launcher — Ana: re-rank the top guesses with the nearest training photos (no training)

Half of the pilot errors had the right answer as the model's second guess. `scripts/rerank_knn.py` re-ranks the top-K candidates with how close the photo is to each candidate's own **training photos** (the m best cosines), not only the class average: `score = log p + beta * knn`. It helps if the deciding detail or look matches some training photo even when the class average does not (look-alike siblings, a class with two looks).

Run on two pilot checkpoints (both trained on their pilot's fit photos only; LoRA r24 no extras):

| Name | Pilot subset | Checkpoint |
|---|---|---|
| `random` | 1,000 random classes (`c1000_h5_s0`) | `A_random_crops` of the training-crop pilot (0.898) |
| `blocks` | 100 blocks of 10 neighbouring classes (`c1000_h5_s0_b10`), relatives present | `A_uniform_smoothing` of the taxonomy pilot (0.835) |

Honest evaluation: K (2/3/5 candidates), m (1/3 nearest photos) and beta are chosen on half the holdout classes and scored on the other half, then swapped; fixes (wrong → right) and breaks (right → wrong) are counted. `before` also gives top-1/2/3/5 and whether errors sit at rank 2 or 3–5, for both subsets.

Outputs in `CS604_RUNS/rerank_knn/<name>/`: `rerank.json`, `rerank_grid.csv`. About 30 min, mostly the data download. Any GPU. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
PILOT_DATA_DIR = "/content/pilot_data"
# (pilot folder, arm, class_block) per subset; run folders are read from the arm records.
RUNS = {
    "random": ("ablation_ablation_train_crops", "A_random_crops", 1),
    "blocks": ("ablation_ablation_taxonomy_smoothing", "A_uniform_smoothing", 10),
}
OUT_NAME = "rerank_knn"
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Re-rank on both pilot subsets

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from scripts.pilot_sweep import build_pilot_root

OUT = Path(DRIVE_OUT) / OUT_NAME
for name, (folder, arm, block) in RUNS.items():
    root, _ = build_pilot_root(DATA_DIR, PILOT_DATA_DIR, 1000, 5, 0, block)
    run_dir = json.loads((Path(DRIVE_OUT) / folder / "arms" / f"{arm}.json").read_text())["run_dir"]
    assert (Path(run_dir) / "last.pt").is_file(), f"no checkpoint in {run_dir}"
    print(f"\n===== {name}: {arm} on {root}")
    if (OUT / name / "rerank.json").is_file():
        print("already done"); continue
    proc = subprocess.run([sys.executable, "-u", "scripts/rerank_knn.py", "--run_dir", run_dir,
                           "--pilot_root", str(root), "--out_dir", str(OUT / name)])
    if proc.returncode != 0:
        end_runtime(f"re-ranking failed on {name} (exit {proc.returncode})")
        raise SystemExit("re-ranking failed")


## 6. Results

- **before**: top-1/2/3/5 and where the truth sits when wrong (rank 2 vs 3–5), per subset.
- **held_out_gain_points**: the gain on the half of the holdout the settings were *not* chosen on (the honest number); **fixes / breaks** per half.
- `best_on_all_photos_optimistic` is tuned and scored on the same photos: an upper bound only.

In [ ]:
import pandas as pd

for name in RUNS:
    s = json.loads((OUT / name / "rerank.json").read_text())
    print(f"===== {name}")
    print("before:", json.dumps(s["before"]))
    display(pd.DataFrame(s["cross_validated"]))
    print("held-out gain (points):", s["held_out_gain_points"])


## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")